# Notebook 03 — Graph Construction

1. Build directed weighted graph from cleaned data
2. Verify node/edge attributes
3. Basic graph statistics
4. node2vec embedding training
5. Save graph for downstream use

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from src.graph.builder import build_graph, save_graph, graph_to_dataframe
from src.graph.node2vec_emb import train_node2vec, save_embeddings, get_node_feature_matrix
from src.data.feature_eng import build_feature_matrix
print('Graph modules loaded')

In [ ]:
# Load data
df = pd.read_parquet('data/processed/delhivery_enriched.parquet') \
     if __import__('os').path.exists('data/processed/delhivery_enriched.parquet') \
     else pd.read_parquet('data/processed/delhivery_clean.parquet')
df = build_feature_matrix(df)
print(f'Dataset: {df.shape}')

In [ ]:
G = build_graph(df)
print(f'Nodes: {G.number_of_nodes()}')
print(f'Edges: {G.number_of_edges()}')
print(f'Strongly connected: {nx.is_strongly_connected(G)}')
print(f'Weakly connected components: {nx.number_weakly_connected_components(G)}')
print(f'Avg out-degree: {sum(dict(G.out_degree()).values()) / G.number_of_nodes():.2f}')
save_graph(G)

In [ ]:
# Degree distribution
degrees = [d for _, d in G.degree()]
plt.figure(figsize=(10,4), facecolor='#0e1117')
plt.hist(degrees, bins=40, color='#4f90ff', edgecolor='none')
plt.title('Hub Degree Distribution', color='white')
plt.xlabel('Degree', color='white'); plt.ylabel('Count', color='white')
plt.tight_layout()
plt.savefig('reports/03_degree_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
plt.show()

In [ ]:
# Edge weight (delay ratio) distribution
edge_weights = [d['median_delay_ratio'] for _,_,d in G.edges(data=True)]
plt.figure(figsize=(10,4), facecolor='#0e1117')
plt.hist(edge_weights, bins=60, color='#ff8844', edgecolor='none')
plt.axvline(1.0, color='green', lw=2, label='On-time')
plt.axvline(1.2, color='red', lw=2, linestyle='--', label='Chronic threshold')
plt.title('Edge Delay Ratio Distribution', color='white')
plt.legend(); plt.tight_layout()
plt.savefig('reports/03_edge_delay_dist.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
plt.show()
chronic_pct = sum(1 for w in edge_weights if w > 1.2) / len(edge_weights) * 100
print(f'Chronic corridors: {chronic_pct:.1f}%')

In [ ]:
# Train node2vec embeddings
embeddings = train_node2vec(G, dimensions=128, walk_length=30, num_walks=100)
save_embeddings(embeddings)
X, node_order = get_node_feature_matrix(G, embeddings)
print(f'Node feature matrix: {X.shape}  (node2vec + handcrafted)')

In [ ]:
# Visualise embeddings with PCA
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
coords = pca.fit_transform(X[:, :128])  # node2vec part only

# Color by hub type if available
hub_types = [G.nodes[n].get('hub_type','unknown') for n in node_order]
unique_types = list(set(hub_types))
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_types)))
color_map = dict(zip(unique_types, colors))

plt.figure(figsize=(12, 8), facecolor='#0e1117')
for ht in unique_types:
    mask = [i for i, h in enumerate(hub_types) if h == ht]
    plt.scatter(coords[mask,0], coords[mask,1], s=20, label=ht, alpha=0.7, c=[color_map[ht]])
plt.legend(fontsize=8); plt.title('node2vec Embeddings (PCA 2D)', color='white')
plt.tight_layout()
plt.savefig('reports/03_node2vec_pca.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
plt.show()